# NB03 streaming ASR eval (Jalur A)
sherpa-onnx streaming zipformer en-2023-06-26 vs Vosk small en-us 0.15, on own clips.
Each WAV is fed in 100 ms chunks at real-time pace. We log the wall-clock time of every partial.

- WER: `jiwer`, same normalization for both engines (lowercase, strip punctuation).
- first-partial latency: stream start to first non-empty partial.
- stable-partial latency: stream start to the first partial that equals the final text and never changes after.
- RTF: decode compute time / audio time (sleep excluded).
- Biasing (zipformer hotwords) only as a second configuration.

Settings: CPU only (Accelerator none), Internet ON. Input: `spellspeak-own-clips`.
EdAcc Indonesian subset: not wired yet, needs a verified way to filter Indonesian speakers (docs/DATASETS.md).

In [ ]:
# Repo code (per.py, words.json). Kaggle cannot see your laptop, so pick one:
#   A) public GitHub repo + Internet ON in notebook settings -> git clone below
#   B) private repo -> upload the repo folder as a private Kaggle Dataset "spellspeak-code" and add it here
import json, os, platform, subprocess, sys, time
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path

REPO_URL = "https://github.com/ddrlve/SpellSpeak.git"
ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
CANDIDATES = [Path("/kaggle/input/spellspeak-code"), Path.cwd().parent, WORK / "SpellSpeak"]
REPO = next((p for p in CANDIDATES if (p / "backend" / "per.py").exists()), None)
if REPO is None:
    REPO = WORK / "SpellSpeak"
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
sys.path.insert(0, str(REPO / "backend"))
RESULTS = WORK / "results"
RESULTS.mkdir(parents=True, exist_ok=True)
print("repo:", REPO, "| results:", RESULTS)

In [ ]:
def cpu_name() -> str:
    try:
        for line in open("/proc/cpuinfo"):
            if line.startswith("model name"):
                return line.split(":", 1)[1].strip()
    except OSError:
        pass
    return platform.processor() or platform.machine()


def device_info(force_cpu: bool = False) -> dict:
    name, threads = cpu_name(), os.cpu_count()
    try:
        import torch
        threads = torch.get_num_threads()
        if torch.cuda.is_available() and not force_cpu:
            name = torch.cuda.get_device_name(0)
    except ImportError:
        pass
    return {"name": name, "threads": threads, "platform": platform.platform()}


def versions(*pkgs: str) -> dict:
    out = {}
    for p in pkgs:
        try:
            out[p] = metadata.version(p)
        except metadata.PackageNotFoundError:
            out[p] = None
    return out


def write_result(notebook: str, data: dict, metrics: dict, notes: str, pkgs: tuple = ()) -> Path:
    """Shape from docs/NOTEBOOKS.md. Only values computed in this run go into metrics."""
    out = {"notebook": notebook, "date": datetime.now(timezone.utc).isoformat(timespec="seconds"),
           "device": device_info(), "data": data, "metrics": metrics, "notes": notes,
           "versions": versions(*pkgs)}
    path = RESULTS / f"{notebook}.json"
    path.write_text(json.dumps(out, indent=2, ensure_ascii=False), encoding="utf-8")
    print(json.dumps(out, indent=2, ensure_ascii=False))
    return path

In [ ]:
!pip install -q sherpa-onnx vosk jiwer
!wget -q -nc https://github.com/k2-fsa/sherpa-onnx/releases/download/asr-models/sherpa-onnx-streaming-zipformer-en-2023-06-26.tar.bz2 && tar xjf sherpa-onnx-streaming-zipformer-en-2023-06-26.tar.bz2
!wget -q -nc https://alphacephei.com/vosk/models/vosk-model-small-en-us-0.15.zip && unzip -q -o vosk-model-small-en-us-0.15.zip

In [ ]:
CHUNK_S = 0.1
REAL_TIME = True       # sleep between chunks like a live mic
ASR_THREADS = 1        # match the app: one thread for Jalur A
HOTWORDS_SCORE = 1.5   # only used in the biasing configuration

In [ ]:
# Own recordings: private Kaggle Dataset made from data/own (backend/record_clips.py output):
#   metadata.csv + wav/*.wav, 16 kHz mono. Real consenting speakers only. Never synthetic clips.
import numpy as np
import pandas as pd
import soundfile as sf

OWN = Path("/kaggle/input/spellspeak-own-clips")
if not (OWN / "metadata.csv").exists():
    raise FileNotFoundError(f"{OWN}/metadata.csv not found. Record clips with backend/record_clips.py, "
                            "upload data/own as a private Kaggle Dataset, add it to this notebook.")
meta = pd.read_csv(OWN / "metadata.csv")
assert meta["consent"].astype(str).str.lower().eq("yes").all(), "every row needs consent=yes"


def load_wav(path: Path) -> np.ndarray:
    audio, sr = sf.read(path, dtype="float32")
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != 16000:
        raise ValueError(f"{path}: need 16000 Hz, got {sr}")
    return audio


own_items = [{"id": r.file, "speaker": r.speaker_id, "word": r.word, "audio": load_wav(OWN / r.file)}
             for r in meta.itertuples()]
print(f"own clips: {len(own_items)} | speakers: {meta.speaker_id.nunique()} | words: {meta.word.nunique()}")

In [ ]:
import re
import jiwer
import sherpa_onnx
import vosk

ZDIR = Path("sherpa-onnx-streaming-zipformer-en-2023-06-26")


def pick(pattern: str) -> str:
    hits = sorted(ZDIR.glob(pattern))
    int8 = [h for h in hits if ".int8." in h.name]
    return str((int8 or hits)[0])


def norm(text: str) -> str:
    return " ".join(re.sub(r"[^a-z' ]", " ", text.lower()).split())


def zipformer(hotwords: list[str] | None = None):
    kw = dict(tokens=str(ZDIR / "tokens.txt"), encoder=pick("encoder-*.onnx"), decoder=pick("decoder-*.onnx"),
              joiner=pick("joiner-*.onnx"), num_threads=ASR_THREADS, sample_rate=16000, feature_dim=80)
    if hotwords:
        hw = WORK / "hotwords.txt"
        hw.write_text("\n".join(w.upper() for w in hotwords), encoding="utf-8")
        kw.update(decoding_method="modified_beam_search", hotwords_file=str(hw), hotwords_score=HOTWORDS_SCORE,
                  modeling_unit="bpe", bpe_vocab=str(ZDIR / "bpe.vocab"))
    else:
        kw.update(decoding_method="greedy_search")
    return sherpa_onnx.OnlineRecognizer.from_transducer(**kw)


def text_of(result) -> str:
    return result if isinstance(result, str) else getattr(result, "text", str(result))


def stream_zipformer(rec, audio: np.ndarray) -> dict:
    s, step, partials, compute = rec.create_stream(), int(16000 * CHUNK_S), [], 0.0
    t_start = time.perf_counter()
    for i in range(0, len(audio), step):
        if REAL_TIME:
            time.sleep(max(0.0, t_start + (i + step) / 16000 - time.perf_counter()))
        t = time.perf_counter()
        s.accept_waveform(16000, audio[i:i + step])
        while rec.is_ready(s):
            rec.decode_stream(s)
        compute += time.perf_counter() - t
        partials.append((time.perf_counter() - t_start, text_of(rec.get_result(s))))
    t = time.perf_counter()
    s.input_finished()
    while rec.is_ready(s):
        rec.decode_stream(s)
    compute += time.perf_counter() - t
    return {"final": text_of(rec.get_result(s)), "partials": partials, "compute_s": compute}


def stream_vosk(model, audio: np.ndarray) -> dict:
    rec = vosk.KaldiRecognizer(model, 16000)
    pcm = (np.clip(audio, -1, 1) * 32767).astype(np.int16)
    step, partials, compute, finals = int(16000 * CHUNK_S), [], 0.0, []
    t_start = time.perf_counter()
    for i in range(0, len(pcm), step):
        if REAL_TIME:
            time.sleep(max(0.0, t_start + (i + step) / 16000 - time.perf_counter()))
        t = time.perf_counter()
        if rec.AcceptWaveform(pcm[i:i + step].tobytes()):
            finals.append(json.loads(rec.Result()).get("text", ""))
            text = " ".join(finals)
        else:
            text = " ".join(finals + [json.loads(rec.PartialResult()).get("partial", "")])
        compute += time.perf_counter() - t
        partials.append((time.perf_counter() - t_start, text))
    t = time.perf_counter()
    finals.append(json.loads(rec.FinalResult()).get("text", ""))
    compute += time.perf_counter() - t
    return {"final": " ".join(f for f in finals if f), "partials": partials, "compute_s": compute}


def latencies(partials: list[tuple[float, str]], final: str) -> tuple[float | None, float | None]:
    first = next((t for t, p in partials if norm(p)), None)
    stable = None
    for t, p in reversed(partials):
        if norm(p) != norm(final):
            break
        stable = t
    return first, stable

In [ ]:
words = [w["word"] for w in json.loads((REPO / "backend" / "words.json").read_text(encoding="utf-8"))["words"]]
engines = {"zipformer_greedy": lambda a: stream_zipformer(zrec, a), "vosk_small": lambda a: stream_vosk(vmodel, a)}
zrec = zipformer()
vmodel = vosk.Model("vosk-model-small-en-us-0.15")
bias_error = None
try:
    zrec_bias = zipformer(hotwords=words)
    engines["zipformer_hotwords"] = lambda a: stream_zipformer(zrec_bias, a)
except Exception as e:  # report, never fake
    bias_error = f"{type(e).__name__}: {e}"
    print("biasing config skipped:", bias_error)

rows = []
for name, run in engines.items():
    run(own_items[0]["audio"])  # warm-up, excluded
    for it in own_items:
        out = run(it["audio"])
        first, stable = latencies(out["partials"], out["final"])
        dur = len(it["audio"]) / 16000
        rows.append({"engine": name, "id": it["id"], "speaker": it["speaker"], "ref": norm(it["word"]),
                     "hyp": norm(out["final"]), "first_partial_s": first, "stable_partial_s": stable,
                     "audio_s": dur, "compute_s": out["compute_s"], "rtf": out["compute_s"] / dur})
asr_df = pd.DataFrame(rows)
asr_df.to_csv(RESULTS / "NB03_per_clip.csv", index=False)
asr_df.head()

In [ ]:
def summarize(g: pd.DataFrame) -> dict:
    return {"n": len(g), "speakers": int(g.speaker.nunique()),
            "wer": jiwer.wer(g.ref.tolist(), [h if h else "<empty>" for h in g.hyp.tolist()]),
            "first_partial_s_median": g.first_partial_s.median(), "first_partial_s_p95": g.first_partial_s.quantile(0.95),
            "stable_partial_s_median": g.stable_partial_s.median(), "stable_partial_s_p95": g.stable_partial_s.quantile(0.95),
            "rtf_median": g.rtf.median(), "rtf_p95": g.rtf.quantile(0.95), "empty_hyp": int((g.hyp == "").sum())}


metrics = {e: {k: (None if pd.isna(v) else float(v)) for k, v in summarize(g).items()} for e, g in asr_df.groupby("engine")}
write_result(
    "NB03_asr_stream_eval",
    data={"source": "own clips", "n_utterances": len(own_items), "speakers": int(meta.speaker_id.nunique())},
    metrics=metrics,
    notes=f"chunk={CHUNK_S}s, real_time={REAL_TIME}, threads={ASR_THREADS}, hotwords_score={HOTWORDS_SCORE}. "
          f"Latencies are seconds from stream start (clip start), include leading silence. "
          f"Biasing error: {bias_error}. EdAcc not run. Kaggle CPU is not the laptop CPU.",
    pkgs=("sherpa-onnx", "vosk", "jiwer"),
)